In [1]:
import cv2
import numpy as np
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecVideoRecorder

In [6]:
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

In [3]:
class BlurBreakout(gym.ObservationWrapper):
    """Aplica desfoque gaussiano ao topo da observação de Breakout."""

    def __init__(self, env, blur_rows=50, kernel_size=5):
        super().__init__(env)
        self.blur_rows = blur_rows
        self.kernel_size = kernel_size
        # a observação continua tendo o mesmo formato (Box)
        self.observation_space = env.observation_space

    def observation(self, obs):
        # Seleciona a região a ser desfocada (linhas iniciais)
        top = obs[: self.blur_rows, :, :]
        blurred_top = cv2.GaussianBlur(top, (self.kernel_size, self.kernel_size), 0)
        obs[: self.blur_rows, :, :] = blurred_top
        return obs


def make_env(with_blur=False):
    env = gym.make("ALE/Breakout-v5")
    # Pre‑processa frames (grayscale, resize etc.) se desejar
    # env = gym.wrappers.AtariPreprocessing(env, grayscale_obs=True, scale_obs=True)
    if with_blur:
        env = BlurBreakout(env, blur_rows=80, kernel_size=7)
    # Registra vídeo e recompensas
    env = gym.wrappers.RecordEpisodeStatistics(env)
    return env

In [ ]:
env_no_blur = make_env(with_blur=False)
# Ambiente com blur
env_blur = make_env(with_blur=True)

# Treina no ambiente sem blur
model_no_blur = DQN(
    policy="CnnPolicy",
    env=env_no_blur,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    gamma=0.99,
    target_update_interval=10_000,
    train_freq=4,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    verbose=1,
)
model_no_blur.learn(total_timesteps=500_000)

# Treina no ambiente com blur
model_blur = DQN(
    policy="CnnPolicy",
    env=env_blur,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    gamma=0.99,
    target_update_interval=10_000,
    train_freq=4,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    verbose=1,
)
model_blur.learn(total_timesteps=500_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


/home/hartb/Estudos/BlurRL/.venv/lib/python3.11/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/hartb/Estudos/BlurRL/.venv/lib/python3.11/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 20.16GB > 4.12GB
  warnings.warn(


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 197      |
|    ep_rew_mean      | 1.5      |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1312     |
|    time_elapsed     | 0        |
|    total_timesteps  | 788      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 217      |
|    ep_rew_mean      | 2        |
|    exploration_rate | 0.966    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1251     |
|    time_elapsed     | 1        |
|    total_timesteps  | 1739     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 205      |
|    ep_rew_mean      | 1.75     |
|    exploration_rate | 0.951    |
| time/               |          |
|    episodes       

In [ ]:
eval_env = make_env(with_blur=True)
eval_env = Monitor(eval_env)

# Grava um episódio
video_env = VecVideoRecorder(
    eval_env,
    video_folder="./videos_blur/",
    record_video_trigger=lambda step: step == 0,
    video_length=500,
    name_prefix="dqn_blur",
)

obs = video_env.reset()
done = False
while not done:
    action, _ = model_blur.predict(obs, deterministic=True)
    obs, reward, done, info = video_env.step(action)